# 🔬 Cryo-EM 3D Reconstruction: A Computational Tutorial

---

## Overview

Cryo-Electron Microscopy (Cryo-EM) has undergone a **resolution revolution** since ~2013,
driven by direct electron detectors and advanced image processing algorithms.

This notebook builds the computational pipeline from first principles:

| Module | Topic |
|--------|-------|
| **1**   | Forward Model: Projections, CTF, Dose Weighting & Noise |
| **1.5** | CTF Correction: Wiener Filter & Phase-Flip |
| **1.8** | 2D Class Averaging (K-means + NCC alignment) |
| **2**   | Fourier Slice Theorem, Trilinear Gridding, B-sharpening, SSNR & Guinier |
| **3**   | Pose Estimation via Expectation-Maximization |
| **4**   | Neural Rendering for Conformational Heterogeneity |

### The Core Imaging Equation

$$I(\mathbf{x}) = \underbrace{\text{CTF}(\mathbf{k})}_{\text{contrast transfer}} * \underbrace{P_R(\rho)(\mathbf{x})}_{\text{2D projection}} + \underbrace{\eta(\mathbf{x})}_{\text{noise}}$$

> **Prerequisites:** `numpy`, `scipy`, `matplotlib`. All cells are self-contained.


In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# pearsonr is imported here (not inline) to avoid NameError
# if cells are run out of order.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import rotate, gaussian_filter
from scipy.fft import fftshift
from scipy.stats import pearsonr          # ← moved to top-level
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.linestyle':   '--',      'image.cmap':      'inferno',
    'font.family':      'monospace','axes.titlesize':  12,
    'axes.labelsize':   10,
})

print("✅ Environment ready. NumPy:", np.__version__)

---
# Module 1 — Forward Projection, CTF, Dose Weighting & Noise

## 1.1 The Physical Imaging Model

Under the **weak-phase object approximation (WPOA)**, the image recorded on the detector is:

$$I(\mathbf{x}) = \text{CTF} * P_R[V](\mathbf{x}) + \eta(\mathbf{x})$$

## 1.2 Contrast Transfer Function (CTF)

$$\text{CTF}(s) = -A\sin[\chi(s)] + \sqrt{1-A^2}\cos[\chi(s)]$$
$$\chi(s) = \pi \lambda s^2 \!\left(\Delta z - \tfrac{1}{2}\lambda^2 s^2 C_s\right)$$

Key parameters: defocus $\Delta z$, spherical aberration $C_s$, wavelength $\lambda$, amplitude contrast $A$.

The CTF oscillates through **zero-crossings** (Thon rings), destroying information at
specific spatial frequencies. Real experiments collect multiple defoci so their zeros
are at different positions, enabling complete Fourier coverage.

## 1.3 Astigmatism

Real microscopes always have some **astigmatism**: the defocus differs along two
perpendicular axes $(\Delta z_u, \Delta z_v)$, making the CTF elliptically anisotropic.
CtfFind / GCTF fit both principal defoci from the Thon ring pattern.

## 1.4 Dose Weighting(Dose-weighting is a key signal processing method adopted to solve the problem of electron beam radiation damage)


Movies are collected as $F$ frames; radiation damage progressively destroys
high-frequency signal in later frames. The **critical dose** $q_c(s)$ is the
accumulated dose (e⁻/Å²) at which the amplitude at frequency $s$ falls to $1/e$:

$$q_c(s) = a \cdot s^b + c \qquad (a=0.245,\ b=-1.665,\ c=2.81 \text{ for 300 kV})$$

Frame $f$ with accumulated dose $D_f$ receives weight $\exp\!\left(-D_f/2q_c(s)\right)$.
Note: using $b=-0.5$ (i.e., $1/\sqrt{s}$) as sometimes quoted is a significant
approximation — the actual fit exponent is $-1.665$.

## 1.5 Noise Model

Cryo-EM images have SNR ~ 0.01–0.1 due to shot noise, inelastic scattering, and
detector DQE. We model the combined effect as additive Gaussian noise.

In [ ]:
# ============================================================
# MODULE 1 — SYNTHETIC PHANTOM & FORWARD MODEL
# ============================================================

def build_phantom(N=64):
    """
    Construct an asymmetric 3D electron density phantom.
    Asymmetry is deliberate: symmetric phantoms make pose
    estimation degenerate.
    Components: core + two lobes + flexible arm.
    """
    c = np.linspace(-1.0, 1.0, N)
    Z, Y, X = np.meshgrid(c, c, c, indexing='ij')

    def gauss3d(x0, y0, z0, sx, sy, sz, amp=1.0):
        return amp * np.exp(-0.5*(
            ((X-x0)/sx)**2 + ((Y-y0)/sy)**2 + ((Z-z0)/sz)**2))

    vol  = gauss3d( 0.00,  0.00,  0.00, 0.35, 0.35, 0.30, 1.0)  # core
    vol += gauss3d( 0.40,  0.20, -0.10, 0.18, 0.18, 0.20, 0.7)  # lobe A
    vol += gauss3d(-0.35, -0.25,  0.15, 0.15, 0.22, 0.15, 0.5)  # lobe B
    vol += gauss3d( 0.55, -0.40,  0.30, 0.08, 0.08, 0.25, 0.4)  # arm
    vol = (vol - vol.min()) / (vol.max() - vol.min())
    return vol


def project_volume(volume, euler_angles):
    """
    2D projection: rotate volume (ZYZ Euler, degrees) then sum along Z.
    Uses scipy.ndimage.rotate (trilinear interpolation internally).
    Note: boundary artifacts are suppressed by the real-space mask
    applied during reconstruction.
    """
    phi, theta, psi = euler_angles
    rot = rotate(volume, phi,   axes=(1, 2), reshape=False)
    rot = rotate(rot,   theta,  axes=(0, 2), reshape=False)
    rot = rotate(rot,   psi,    axes=(1, 2), reshape=False)
    return rot.sum(axis=0)


def compute_ctf(N, pixel_size_A=3.0, defocus_um=-2.0,
                Cs_mm=2.7, voltage_kV=300, amp_contrast=0.07,
                bfactor=0.0):
    """
    2D isotropic CTF.
    FIX: indexing='ij' so (fx[i,j], fy[i,j]) matches np.fft.fft2[i,j].
    Using 'xy' is semantically wrong and breaks astigmatic extensions.
    """
    lam_A = (12.2639 / np.sqrt(voltage_kV*1e3*(1 + voltage_kV/1022))) * 1e-10 * 1e10
    Cs_A  = Cs_mm * 1e7
    dz_A  = defocus_um * 1e4

    freqs = np.fft.fftfreq(N, d=pixel_size_A)
    # ✅ FIX: was indexing='xy', now 'ij' for correct Fourier alignment
    fx, fy = np.meshgrid(freqs, freqs, indexing='ij')
    s = np.sqrt(fx**2 + fy**2)

    chi = np.pi * lam_A * s**2 * (dz_A - 0.5 * lam_A**2 * s**2 * Cs_A)
    A   = amp_contrast
    ctf = -A * np.sin(chi) + np.sqrt(1 - A**2) * np.cos(chi)

    if bfactor > 0:
        ctf *= np.exp(-bfactor * s**2 / 4.0)
    return ctf, np.abs(freqs[:N//2])


def compute_ctf_anisotropic(N, pixel_size_A=3.0,
                             defocus_u=-2.0, defocus_v=-2.4,
                             astigmatism_angle_deg=30.0,
                             Cs_mm=2.7, voltage_kV=300, amp_contrast=0.07):
    """
    2D CTF with astigmatism — two distinct principal defoci.
    All real microscope images have some astigmatism; CtfFind/GCTF
    fits both defocus_u and defocus_v from the Thon ring ellipticity.

    defocus_u, defocus_v        : principal defoci in μm
    astigmatism_angle_deg       : orientation of slow (less-defocused) axis
    """
    lam_A = (12.2639 / np.sqrt(voltage_kV*1e3*(1 + voltage_kV/1022))) * 1e-10 * 1e10
    Cs_A  = Cs_mm * 1e7
    ang   = np.deg2rad(astigmatism_angle_deg)

    freqs  = np.fft.fftfreq(N, d=pixel_size_A)
    fx, fy = np.meshgrid(freqs, freqs, indexing='ij')
    phi    = np.arctan2(fy, fx)

    # Defocus varies as a cosine around the astigmatic axes
    dz_A  = 0.5 * ((defocus_u + defocus_v)*1e4 +
                    (defocus_u - defocus_v)*1e4 * np.cos(2*(phi - ang)))
    s2    = fx**2 + fy**2
    chi   = np.pi * lam_A * s2 * (dz_A - 0.5 * lam_A**2 * s2 * Cs_A)
    A     = amp_contrast
    return -A * np.sin(chi) + np.sqrt(1 - A**2) * np.cos(chi)


def apply_ctf(projection, ctf):
    """Convolve a 2D projection with the CTF in Fourier space."""
    return np.real(np.fft.ifft2(np.fft.fft2(projection) * ctf))


def add_noise(image, snr=0.05):
    """
    Add additive white Gaussian noise at a given linear SNR.
    noise_std = signal_rms / sqrt(SNR)
    """
    noise_std = np.sqrt(np.mean(image**2) / snr)
    return image + np.random.normal(0.0, noise_std, image.shape)

#In actual collection, the electronic dose is applied in multiple frames. The greater the cumulative dose, the more severe the radiation damage and the faster the high-frequency signal attenuation. Therefore, modern single-particle analysis typically employs dose weighting per frame to optimize the reconstruction quality.
def dose_weight_filter(N, pixel_size_A, total_dose_epA2, n_frames):
    """
    Per-frame dose weighting filter (Grant & Grigorieff 2015).

    CORRECTED formula: q_crit(s) = a * s^b + c
      a = 0.245, b = -1.665, c = 2.81  (empirical fit at 300 kV)

    Note: some sources quote s^{-0.5} but the actual fit exponent
    is -1.665, which matters significantly at high spatial frequency.

    Parameters
    ----------
    total_dose_epA2 : float — total dose in e⁻/Å²
    n_frames        : int   — number of movie frames

    Returns
    -------
    weights : (n_frames, N, N) float array
    """
    freqs  = np.fft.fftfreq(N, d=pixel_size_A)
    fx, fy = np.meshgrid(freqs, freqs, indexing='ij')
    s      = np.sqrt(fx**2 + fy**2) + 1e-8   # avoid divide-by-zero

    # Grant & Grigorieff 2015, eqn 3 — correct parameterisation
    q_crit = 0.245 * s**(-1.665) + 2.81      # e⁻/Å² at which SNR → 1/e

    dose_per_frame = total_dose_epA2 / n_frames
    weights = []
    for f in range(1, n_frames + 1):
        D_f    = f * dose_per_frame
        w      = np.exp(-D_f / (2.0 * q_crit))
        weights.append(w)
    return np.array(weights)   # (n_frames, N, N)


# ── Build phantom & generate projections ──────────────────────
N       = 64
phantom = build_phantom(N)

true_poses = [
    (  0,   0,   0), ( 45,  30,   0), ( 90,  60,  20),
    (120,  45,  45), (200,  80,  90), (270, 110, 135),
]

CTF_2D, freq_1d   = compute_ctf(N, pixel_size_A=3.0, defocus_um=-2.5)
CTF_ASTIG         = compute_ctf_anisotropic(N, defocus_u=-2.0, defocus_v=-2.8,
                                             astigmatism_angle_deg=45.0)

projections_clean = []
projections_ctf   = []
projections_noisy = []

for pose in true_poses:
    p  = project_volume(phantom, pose)
    pc = apply_ctf(p, CTF_2D)
    pn = add_noise(pc, snr=0.08)
    projections_clean.append(p)
    projections_ctf.append(pc)
    projections_noisy.append(pn)

# Dose-weighting demo (5-frame movie for projection 0)
dose_filters = dose_weight_filter(N, pixel_size_A=3.0,
                                  total_dose_epA2=40.0, n_frames=5)

print(f"✅ Phantom built ({N}³), {len(true_poses)} projections generated.")
print(f"   CTF range: [{CTF_2D.min():.3f}, {CTF_2D.max():.3f}]")
print(f"   Dose weight shape: {dose_filters.shape}")
print(f"   Frame-1 weight at Nyquist: {dose_filters[0, N//2, N//2]:.4f}")
print(f"   Frame-5 weight at Nyquist: {dose_filters[4, N//2, N//2]:.4f}")

### Why the CTF effect and noise look "invisible" — and why that is correct

#### CTF subtlety

The phantom consists of broad Gaussians. Its **Fourier bandwidth** (3σ of the
largest blob) is only **0.014 Å⁻¹**. The CTF at Δz = −2.5 μm has its first
zero at **0.031 Å⁻¹** — so almost all of the phantom's energy sits in the
CTF's flat region (CTF ≈ 0.9995 there). The convolution barely changes the image.

To make the CTF effect **visually obvious** two things are needed:
1. Use a **large defocus** (Δz = −12 μm) so the first zero at 0.014 Å⁻¹ falls
   inside the phantom's bandwidth.
2. Show the **amplified residual** `(CTF − clean) × 5` — even then the effect
   is subtle on smooth objects, which is exactly why the **Thon ring power
   spectrum** (not the image itself) is used to measure the CTF in practice.

#### Noise burial

With SNR = 0.08, noise_std = **3.54 × signal_RMS**. The six views produce six
distinct projection *shapes*, but since noise >> signal, all six images appear
as indistinguishable white noise. This is **intentional** and physically
correct — it is exactly why cryo-EM requires averaging 10⁴–10⁶ particles.

The visualization below shows:
- A **high-defocus CTF** (Δz = −12 μm) to make the effect detectable
- An **amplified difference row** to make the subtle modulation explicit
- An **SNR ramp** (SNR = 1.0 → 0.2 → 0.05) so you can see the signal
  gradually disappear into noise

In [ ]:
# ============================================================
# VISUALIZATION 1A — (REVISED) Pipeline with diagnostic panels
#
# Changes vs previous version:
#   - CTF row now uses Δz = −12 μm so the first CTF zero (0.014 Å⁻¹)
#     falls inside the phantom's Fourier bandwidth (core 3σ = 0.014 Å⁻¹).
#     At Δz = −2.5 μm the first zero (0.031 Å⁻¹) was above almost all
#     of the phantom's energy, making the CTF invisible.
#   - Added CTF difference row (residual × 5) to make the modulation explicit.
#   - Noise row replaced by a 3-level SNR ramp (1.0 / 0.2 / 0.05) to show
#     progressive signal burial. All three use the SAME random seed per view
#     so the signal-dependent differences are visible at high SNR.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import rotate as _rotate

# ── Generate high-defocus CTF for visualization ───────────────────────────────
CTF_viz, _ = compute_ctf(N, pixel_size_A=3.0, defocus_um=-12.0)

# High-defocus CTF projections (used ONLY for this visualization)
proj_ctf_viz = [apply_ctf(p, CTF_viz) for p in projections_clean]
proj_diff_viz = [(pc - pc_clean) for pc, pc_clean
                  in zip(proj_ctf_viz, projections_clean)]

# SNR ramp — use a fixed seed per view so the signal pattern is reproducible
VIEWS = [0, 1, 2, 3, 4, 5]   # all 6 views
snr_levels = [1.0, 0.2, 0.05]
ramp_stacks = {}
for snr in snr_levels:
    rng = np.random.default_rng(seed=42)
    stack = []
    for pc in proj_ctf_viz:           # apply noise on top of CTF image
        sig_rms   = np.sqrt(np.mean(pc**2))
        noise_std = sig_rms / np.sqrt(snr)
        stack.append(pc + rng.normal(0, noise_std, pc.shape))
    ramp_stacks[snr] = stack

# ── Build figure ──────────────────────────────────────────────────────────────
n_views = 6
fig = plt.figure(figsize=(22, 18))
fig.suptitle('Module 1 — Forward Imaging Pipeline\n'
             'Defocus Δz = −12 μm for CTF rows; SNR ramp shows signal burial',
             fontsize=13, fontweight='bold', color='#58a6ff')

gs = gridspec.GridSpec(6, n_views, figure=fig, hspace=0.55, wspace=0.28)

row_configs = [
    (projections_clean,   'Clean\nprojection',     '#3fb950', 'gray',  None,   None),
    (proj_ctf_viz,         'After CTF\n(Δz=−12μm)', '#79c0ff', 'gray',  None,   None),
    (proj_diff_viz,        'CTF residual\n×5 amp.',  '#d2a8ff', 'RdBu_r',5.0,  True),
    (ramp_stacks[1.0],    'Noise: SNR=1.0\n(signal visible)',   '#8957e5','gray',None,None),
    (ramp_stacks[0.2],    'Noise: SNR=0.2\n(signal faint)',     '#f0883e','gray',None,None),
    (ramp_stacks[0.05],   'Noise: SNR=0.05\n(signal buried)',   '#ff7b72','gray',None,None),
]

for ri, (stack, label, border_col, cmap, amp, is_diff) in enumerate(row_configs):
    for ci, img in enumerate(stack):
        ax = fig.add_subplot(gs[ri, ci])

        if is_diff:
            # Amplify the residual and use diverging colormap centred at 0
            disp  = img * amp
            vmax  = np.abs(disp).max() * 0.8
            ax.imshow(disp, cmap=cmap, vmin=-vmax, vmax=vmax)
        else:
            ax.imshow(img, cmap=cmap)

        if ci == 0:
            ax.set_ylabel(label, color=border_col, fontsize=8,
                          fontweight='bold', labelpad=4)
        ax.set_title(f'φ={true_poses[ci][0]}°', fontsize=7, color='#8b949e')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(border_col); sp.set_linewidth(1.8)

# ── Annotation: mark the CTF first-zero frequency ─────────────────────────────
# Add a small inset on the CTF row showing a 1D CTF profile with the
# phantom's bandwidth marked
ax_inset = fig.add_axes([0.88, 0.68, 0.10, 0.08])   # upper-right inset
lam_A_loc = (12.2639/np.sqrt(300e3*(1+300/1022)))*1e-10*1e10
s1d = np.linspace(0, 1/(2*3.0), 300)
for dz, col, ls in [(-2.5,'#79c0ff','--'), (-12.0,'#d2a8ff','-')]:
    dz_A = dz*1e4
    chi  = np.pi*lam_A_loc*s1d**2*(dz_A - 0.5*lam_A_loc**2*s1d**2*2.7e7)
    ctf1 = -0.07*np.sin(chi)+np.sqrt(1-0.07**2)*np.cos(chi)
    ax_inset.plot(s1d, ctf1, color=col, lw=1.2, ls=ls,
                  label=f'{dz} μm')
ax_inset.axvline(0.014, color='#f0883e', lw=1.0, ls=':', label='Phantom 3σ')
ax_inset.axhline(0, color='#444', lw=0.6)
ax_inset.set_title('CTF vs\nbandwidth', fontsize=6, color='#e6edf3')
ax_inset.tick_params(labelsize=5, colors='#8b949e')
ax_inset.legend(fontsize=4.5, loc='upper right')
ax_inset.set_facecolor('#161b22')
for sp in ax_inset.spines.values(): sp.set_edgecolor('#30363d')

plt.tight_layout(rect=[0, 0, 1, 0.97])
from pathlib import Path
out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'viz1a_revised.png'
plt.savefig(out_path, dpi=140, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n📊 Key diagnostics (figure saved to {out_path.resolve()}):")
print(f"  CTF first zero at Δz=−2.5μm : 0.031 Å⁻¹  (ABOVE phantom 3σ=0.014 Å⁻¹)")
print(f"  CTF first zero at Δz=−12μm  : 0.014 Å⁻¹  (AT phantom 3σ → effect visible)")
print(f"  SNR=1.0 : noise = 1.00× signal_RMS")
print(f"  SNR=0.2 : noise = 2.24× signal_RMS")
print(f"  SNR=0.05: noise = 4.47× signal_RMS  ← real cryo-EM conditions")


In [ ]:
# ============================================================
# VISUALIZATION 1B — Isotropic vs Anisotropic CTF & Dose Weighting
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('CTF Variants & Dose Weighting',
             fontsize=14, fontweight='bold', color='#58a6ff')

# Isotropic CTF
im = axes[0,0].imshow(fftshift(CTF_2D), cmap='RdBu_r', vmin=-1, vmax=1)
axes[0,0].set_title('Isotropic CTF\n(Δz=−2.5 μm)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,0], fraction=0.046)

# Isotropic CTF² (Thon rings)
im = axes[0,1].imshow(fftshift(CTF_2D**2), cmap='viridis')
axes[0,1].set_title('CTF² — Thon rings\n(isotropic)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,1], fraction=0.046)

# Anisotropic CTF² (elliptic Thon rings)
im = axes[0,2].imshow(fftshift(CTF_ASTIG**2), cmap='viridis')
axes[0,2].set_title('CTF² — Thon rings\n(astigmatic, Δu=−2.0, Δv=−2.8 μm)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,2], fraction=0.046)

# Dose weighting: compare s^{-0.5} approx vs actual s^{-1.665}
freqs1d = np.linspace(0.001, 1/(2*3.0), 300)
q_approx = 0.245 / np.sqrt(freqs1d)           # wrong approximation
q_correct = 0.245 * freqs1d**(-1.665) + 2.81  # Grant & Grigorieff 2015

axes[1,0].semilogy(freqs1d, q_approx,  color='#ff7b72', lw=2, label='s^{-0.5} (approx)')
axes[1,0].semilogy(freqs1d, q_correct, color='#3fb950',  lw=2, label='s^{-1.665}+2.81 (correct)')
axes[1,0].set_title('Critical dose q_c(s)\nGrant & Grigorieff 2015', color='#e6edf3')
axes[1,0].set_xlabel('Spatial freq (1/Å)'); axes[1,0].set_ylabel('q_crit (e⁻/Å²)')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True)

# Per-frame dose weights vs frequency
for f_idx, col in enumerate(['#79c0ff','#3fb950','#f0883e','#ff7b72','#d2a8ff']):
    w_radial = dose_filters[f_idx, :N//2, N//2]  # radial slice
    axes[1,1].plot(freq_1d, w_radial, color=col, lw=1.8,
                   label=f'Frame {f_idx+1}')
axes[1,1].set_title('Per-frame dose weights\n(40 e⁻/Å² total, 5 frames)', color='#e6edf3')
axes[1,1].set_xlabel('Spatial freq (1/Å)'); axes[1,1].set_ylabel('Weight')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True)

# Dose-weighted sum for projection 0 (simulate optimal frame combination)
proj_ftf = np.fft.fft2(projections_clean[0])
dose_weighted_sum = np.zeros_like(proj_ftf)
for f_idx in range(5):
    dose_weighted_sum += dose_filters[f_idx] * proj_ftf
dose_weighted_proj = np.real(np.fft.ifft2(dose_weighted_sum / 5))

axes[1,2].imshow(dose_weighted_proj, cmap='gray')
axes[1,2].set_title('Dose-weighted projection\n(optimal frame combination)', color='#e6edf3')
axes[1,2].set_xticks([]); axes[1,2].set_yticks([])

for row in axes:
    for ax in row:
        if not ax.get_title():
            ax.set_visible(False)

plt.tight_layout(); plt.show()

---
# Module 1.5 — CTF Correction

## Why CTF Correction is Essential

The CTF **zeros** destroy information at specific spatial frequencies.
If we reconstruct using raw CTF-corrupted images, those frequencies will be
absent or sign-flipped in the final map — producing artifacts and loss of resolution.
**Every real pipeline CTF-corrects before or during reconstruction.**

## Phase-Flip Correction

The simplest correction: multiply the Fourier image by $\text{sign}(\text{CTF}(k))$,
flipping the phase of lobes with negative CTF. This restores phase coherence but does
**not** restore the amplitude at CTF zeros — information there remains missing.

$$\hat{I}_{\text{corrected}}(\mathbf{k}) = \hat{I}(\mathbf{k}) \cdot \text{sign}(\text{CTF}(\mathbf{k}))$$

## Wiener Filter (Optimal CTF Correction)

The Wiener filter is the **minimum mean-square-error** inverse filter:

$$H_{\text{Wiener}}(\mathbf{k}) = \frac{\text{CTF}(\mathbf{k})}{\text{CTF}^2(\mathbf{k}) + 1/\text{SNR}}$$

The $1/\text{SNR}$ regulariser prevents catastrophic noise amplification near CTF zeros.
At high SNR (→ ∞) it reduces to a simple inverse; at low SNR it rolls off gracefully.

In practice, SNR is frequency-dependent (from the noise power spectrum), but a
global estimate suffices for this tutorial.

In [ ]:
# ============================================================
# MODULE 1.5 — CTF CORRECTION
# ============================================================

def wiener_ctf_correct(image_ctf, ctf, snr_estimate=0.1):
    """
    Wiener filter CTF correction.
    H_wiener(k) = CTF(k) / (CTF²(k) + 1/SNR)

    The 1/SNR regulariser prevents noise amplification at CTF zeros.
    At SNR→∞ this approaches a pure inverse filter; at low SNR it
    gracefully down-weights unreliable frequencies.
    """
    F_img  = np.fft.fft2(image_ctf)
    wiener = ctf / (ctf**2 + 1.0 / snr_estimate)
    return np.real(np.fft.ifft2(F_img * wiener))


def phase_flip_correct(image_ctf, ctf):
    """
    Phase-flip correction: multiply by sign(CTF) to restore phase coherence.
    Faster but does not restore amplitudes at CTF zeros.
    """
    F_img  = np.fft.fft2(image_ctf)
    return np.real(np.fft.ifft2(F_img * np.sign(ctf)))


projections_wiener    = [wiener_ctf_correct(img, CTF_2D, snr_estimate=0.08)
                         for img in projections_noisy]
projections_phaseflip = [phase_flip_correct(img, CTF_2D)
                         for img in projections_noisy]

print("✅ CTF correction applied (Wiener + phase-flip) to all 6 projections.")

In [ ]:
# ============================================================
# VISUALIZATION 1.5 — CTF Correction Comparison
# ============================================================
fig, axes = plt.subplots(4, 3, figsize=(13, 16))
fig.suptitle('Module 1.5 — CTF Correction Pipeline\n'
             'Rows: raw noisy | phase-flipped | Wiener | power spectra',
             fontsize=13, fontweight='bold', color='#58a6ff')

row_data   = [projections_noisy, projections_phaseflip, projections_wiener]
row_labels = ['Raw noisy (SNR≈0.08)', 'Phase-flip corrected', 'Wiener corrected']
row_colors = ['#ff7b72', '#f0883e', '#3fb950']

for ri,(stack,lab,col) in enumerate(zip(row_data, row_labels, row_colors)):
    for ci in range(3):
        ax = axes[ri, ci]
        ax.imshow(stack[ci], cmap='gray')
        if ci == 0:
            ax.set_ylabel(lab, color=col, fontsize=8, fontweight='bold')
        ax.set_title(f'View {ci+1}', color='#8b949e', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(col); sp.set_linewidth(1.5)

# Row 3: radial power spectra comparing corrections
ax_ps = axes[3, 0]
ax_ps2 = axes[3, 1]
ax_ps3 = axes[3, 2]
for ax_i, (stack, lab, col) in zip([ax_ps, ax_ps2, ax_ps3],
                                    zip(row_data, row_labels, row_colors)):
    ps = np.abs(np.fft.fftshift(np.fft.fft2(stack[0])))**2
    # Radial average
    freqs_r = np.fft.fftfreq(N)
    kx2, ky2 = np.meshgrid(freqs_r, freqs_r, indexing='ij')
    r = np.sqrt(kx2**2 + ky2**2)
    r_idx = np.round(r * N).astype(int)
    radial = np.array([ps[r_idx == s].mean() if (r_idx==s).any() else 0
                        for s in range(N//2)])
    ax_i.semilogy(np.arange(N//2) / N / 3.0, radial + 1e-8,
                  color=col, lw=1.8)
    ax_i.set_title(f'Power spectrum\n{lab}', color=col, fontsize=8)
    ax_i.set_xlabel('Freq (1/Å)', fontsize=8); ax_i.grid(True)
    ax_i.set_ylabel('Power (log)', fontsize=8)

plt.tight_layout(); plt.show()

# Print SNR improvement metric (RMS signal recovery)
gt_flat = projections_clean[0].ravel()
for lab, stack in [('Noisy', projections_noisy),
                   ('Phase-flip', projections_phaseflip),
                   ('Wiener', projections_wiener)]:
    r, _ = pearsonr(gt_flat, stack[0].ravel())
    print(f"  {lab:15s}: Pearson r with clean projection = {r:.4f}")

---
# Module 1.8 — 2D Class Averaging

## Why 2D Classification Precedes 3D Reconstruction

In production pipelines (RELION, cryoSPARC), **2D class averaging** is mandatory before
3D work. It serves three purposes:

1. **Denoising**: averaging $N$ aligned particles with the same view improves SNR by $\sqrt{N}$
2. **Particle screening**: junk picks (ice, carbon edges, aggregates) form diffuse or
   structureless 2D classes and are discarded
3. **Data validation**: clear 2D averages confirm the dataset is interpretable before the
   expensive 3D step

## Algorithm (Simplified K-means with NCC Alignment)

Production software uses EM with rotation-invariant Fourier features + principal components,
but the conceptual core is:

1. **Initialize**: randomly assign particles to $K$ classes; compute class references as means
2. **E-step**: for each particle, find the best in-plane rotation vs each reference (NCC peak),
   assign to best-matching class
3. **M-step**: update each class reference = mean of all assigned (rotated) particles
4. **Repeat** until assignments converge

The NCC peak **position** gives the optimal 2D translation; the peak **value** gives the
similarity score. Taking `cc.max()` implicitly marginalizes over all in-plane shifts.

## Relation to 3D Reconstruction

Each 2D class average corresponds to a **projection direction** (or a small angular range).
After 3D reconstruction, the 3D map re-projected at those orientations should match the
2D class averages — this is the primary quality check used in practice.


In [ ]:
# ============================================================
# MODULE 1.8 — 2D CLASS AVERAGING
# ============================================================
# Define NCC locally here so Module 1.8 has no dependency on
# Module 3 (where ncc_score is formally introduced).

def _ncc2d(image, template):
    """
    Normalised cross-correlation via FFT.
    Returns cc.max() — the peak value after marginalising over
    all in-plane translations. Complexity: O(N² log N).
    """
    F_img = np.fft.fft2(image    - image.mean())
    F_ref = np.fft.fft2(template - template.mean())
    cc    = np.real(np.fft.ifft2(F_img * np.conj(F_ref)))
    norm  = image.std() * template.std() * image.size
    return float(cc.max() / (norm + 1e-8))


def align_to_reference(image, reference, n_angles=24):
    """
    Find the best in-plane rotation of `image` matching `reference`.
    Returns (rotated_image, best_angle, best_ncc_score).
    Coarse search over `n_angles` uniformly-spaced angles.
    """
    best_score, best_angle, best_rot = -np.inf, 0.0, image
    for ang in np.linspace(0, 360, n_angles, endpoint=False):
        rot   = rotate(image, ang, reshape=False)
        score = _ncc2d(rot, reference)
        if score > best_score:
            best_score, best_angle, best_rot = score, ang, rot
    return best_rot, best_angle, best_score


def kmeans_2d_classify(images, n_classes=3, n_iter=5, n_angles=24, seed=0):
    """
    Simple 2D class averaging via iterative K-means with in-plane NCC alignment.

    Parameters
    ----------
    images    : list of (N,N) arrays — CTF-corrected noisy projections
    n_classes : int — number of 2D classes
    n_iter    : int — number of K-means iterations
    n_angles  : int — angular sampling for in-plane search

    Returns
    -------
    class_refs   : list of (N,N) arrays  — class-average images
    assignments  : (n_imgs,) int array   — class index per particle
    ncc_scores   : (n_imgs, n_classes) float array — best NCC per class
    """
    rng    = np.random.default_rng(seed)
    n_imgs = len(images)
    imgs   = np.array(images, dtype=float)

    # Initialise references: random partition
    perm   = rng.permutation(n_imgs)
    refs   = [imgs[perm[i::n_classes]].mean(axis=0) for i in range(n_classes)]

    assignments = np.zeros(n_imgs, dtype=int)

    for iteration in range(n_iter):
        scores  = np.zeros((n_imgs, n_classes))
        aligned = np.zeros_like(imgs)

        for i, img in enumerate(imgs):
            for k, ref in enumerate(refs):
                rot, _, sc  = align_to_reference(img, ref, n_angles=n_angles)
                scores[i, k] = sc
                if k == 0 or sc > scores[i, :k].max():
                    aligned[i] = rot    # keep best-aligned rotation

        new_assignments = np.argmax(scores, axis=1)

        # M-step: update references
        new_refs = []
        for k in range(n_classes):
            members = np.where(new_assignments == k)[0]
            if len(members) == 0:
                new_refs.append(refs[k])          # keep old ref if empty
            else:
                new_refs.append(aligned[members].mean(axis=0))

        changed = np.sum(new_assignments != assignments)
        assignments = new_assignments
        refs = new_refs
        print(f"  Iter {iteration+1}/{n_iter} — {changed} reassignments")
        if changed == 0:
            break

    return refs, assignments, scores


# ── Generate a larger particle set from 3 angle clusters ───────
# 3 clusters × 8 particles each = 24 noisy projections
angle_cluster_centers = [(0, 0, 0), (90, 50, 0), (170, 100, 45)]
n_per_cluster         = 8
np.random.seed(13)

imgs_2d, true_labels_2d = [], []
for k, (phi0, theta0, psi0) in enumerate(angle_cluster_centers):
    for _ in range(n_per_cluster):
        phi   = phi0   + np.random.uniform(-12, 12)
        theta = theta0 + np.random.uniform(-8,   8)
        psi   = psi0   + np.random.uniform(-12, 12)
        proj  = project_volume(phantom, (phi, theta, psi))
        proj_c = apply_ctf(proj, CTF_2D)
        proj_n = add_noise(proj_c, snr=0.10)
        proj_w = wiener_ctf_correct(proj_n, CTF_2D, snr_estimate=0.10)
        imgs_2d.append(proj_w)
        true_labels_2d.append(k)

true_labels_2d = np.array(true_labels_2d)
print(f"✅ Generated {len(imgs_2d)} Wiener-corrected particles from 3 angle clusters.")
print("🔄 Running 2D K-means classification (3 classes, 5 iterations)...")

class_refs, assignments_2d, ncc_mat = kmeans_2d_classify(
    imgs_2d, n_classes=3, n_iter=5, n_angles=24
)

# Contingency table: how well do recovered classes map to true angle clusters?
print("\n✅ 2D classification complete.")
print("   Contingency table (recovered class vs true cluster):")
print("   ", " ".join([f"Cls{k}" for k in range(3)]))
for true_k in range(3):
    row = [np.sum((assignments_2d == k) & (true_labels_2d == true_k)) for k in range(3)]
    print(f"   Cluster {true_k}: {row}")


In [ ]:
# ============================================================
# VISUALIZATION 1.8 — 2D Class Averages & Assignment Quality
# ============================================================
fig = plt.figure(figsize=(20, 12))
fig.suptitle('Module 1.8 — 2D Class Averaging (K-means + NCC alignment)',
             fontsize=14, fontweight='bold', color='#58a6ff')

class_colors = ['#79c0ff', '#3fb950', '#f0883e']
class_labels = [f'Class {k+1}' for k in range(3)]

# Top block: recovered class averages (row 1) + ground-truth means (row 2)
outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 2], hspace=0.28)
gs_top = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=outer[0], hspace=0.35, wspace=0.25)

for k, (ref, col, lab) in enumerate(zip(class_refs, class_colors, class_labels)):
    ax = fig.add_subplot(gs_top[0, k])
    n_members = int((assignments_2d == k).sum())
    ax.imshow(ref, cmap='inferno', interpolation='bilinear')
    ax.set_title(f'{lab} average\n({n_members} particles)',
                 color=col, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(col); sp.set_linewidth(2.5)

for k in range(3):
    members = np.where(true_labels_2d == k)[0]
    true_ref = np.array(imgs_2d)[members].mean(axis=0)
    ax = fig.add_subplot(gs_top[1, k])
    ax.imshow(true_ref, cmap='inferno', interpolation='bilinear')
    ax.set_title(f'True cluster {k+1}\n(GT mean)', color='#d2a8ff', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor('#d2a8ff'); sp.set_linewidth(2.0)

# Bottom block: all particle thumbnails colored by assigned class
gs_thumb = gridspec.GridSpecFromSubplotSpec(3, 8, subplot_spec=outer[1], hspace=0.18, wspace=0.12)

for idx, img in enumerate(imgs_2d):
    r, c = divmod(idx, 8)
    ax = fig.add_subplot(gs_thumb[r, c])
    k_assigned = assignments_2d[idx]
    k_true = true_labels_2d[idx]
    ax.imshow(img, cmap='gray', interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])

    # Border encodes assigned class; dashed border means misassigned
    col_border = class_colors[k_assigned]
    correct = (k_assigned == k_true)
    lw = 2.5 if correct else 1.2
    ls = '-' if correct else '--'
    for sp in ax.spines.values():
        sp.set_edgecolor(col_border)
        sp.set_linewidth(lw)
        sp.set_linestyle(ls)

legend_elements = [
    plt.Line2D([0], [0], color=c, lw=3, label=f'{l} (assigned)')
    for c, l in zip(class_colors, class_labels)
] + [
    plt.Line2D([0], [0], color='w', lw=2.5, linestyle='-', label='Correct assignment'),
    plt.Line2D([0], [0], color='w', lw=1.2, linestyle='--', label='Misassigned'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=9, framealpha=0.2, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.show()

# NCC score distribution
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))
fig2.suptitle('2D Classification — NCC Score Distributions',
              fontsize=12, fontweight='bold', color='#58a6ff')
for k, (ax, col, lab) in enumerate(zip(axes2, class_colors, class_labels)):
    scores_k = ncc_mat[:, k]
    members = assignments_2d == k
    ax.hist(scores_k[members], bins=10, color=col, alpha=0.8, label='Assigned')
    ax.hist(scores_k[~members], bins=10, color='#30363d', alpha=0.6, label='Not assigned')
    ax.set_title(f'{lab} NCC scores', color=col, fontsize=10)
    ax.set_xlabel('NCC score'); ax.set_ylabel('Count')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Module 2 — Fourier Slice Theorem & 3D Reconstruction

## 2.1 The Central Slice Theorem

$$\boxed{\mathcal{F}_2\{P_\theta[\rho]\}(\mathbf{k}_\perp) = \mathcal{F}_3\{\rho\}(R_\theta \mathbf{k}_\perp)}$$

When performing a two-dimensional Fourier transform (2D FFT) on this two-dimensional projection image, the result is completely equivalent to a two-dimensional slice captured at the center of the three-dimensional frequency domain after performing a three-dimensional Fourier transform (3D FFT) on the original three-dimensional object

Each projection provides one oriented **central slice** of the 3D Fourier volume.

## 2.2 Direct Fourier Insertion with Trilinear Gridding

Nearest-neighbour gridding introduces aliasing artifacts that degrade resolution.
**Trilinear (8-point) interpolation** distributes each complex Fourier value across
its 8 surrounding grid neighbours, weighted by fractional distances — dramatically
improving reconstruction fidelity.

The full algorithm:
1. For each CTF-corrected projection $\hat{I}_i$: compute $\mathcal{F}_2\{\hat{I}_i\}$
2. Rotate the 2D frequency plane by $R_i$ into 3D Fourier space
3. **Trilinear scatter** into $F_\text{3D}$ and weight map $W_\text{3D}$
4. Density compensation: $F_\text{3D}^{\text{comp}} = F_\text{3D} / (W_\text{3D} + \epsilon)$
5. Apply 3D spherical mask in real space
6. $\rho = \mathcal{F}_3^{-1}\{F_\text{3D}^{\text{comp}}\}$

## 2.3 3D Spherical Mask

A **soft-edged spherical mask** applied after each reconstruction suppresses solvent
noise outside the particle. Without masking, the FSC is inflated by correlated noise
at low resolution — giving an overly optimistic resolution estimate.

## 2.4 Filtered Back-Projection (FBP)

$$\rho(\mathbf{x}) = \int_0^\pi \mathcal{F}_1^{-1}\{|k| \cdot \mathcal{F}_1\{P_\theta\}\}(\mathbf{x}\cdot\hat{n}_\theta)\,d\theta$$

The ramp filter $|k|$ compensates for polar over-sampling. Windowed (Hann × ramp)
to suppress high-frequency noise amplification.

## 2.5 B-factor Sharpening

Final maps are sharpened by a negative B-factor to counteract the Guinier falloff
from radiation damage, beam coherence, and detector DQE:

$$F_\text{sharp}(\mathbf{k}) = F(\mathbf{k}) \cdot e^{B \cdot s^2 / 4}, \quad B < 0$$

Always paired with a low-pass filter to avoid boosting noise above the resolution limit.

## 2.6 Inter-module note: 2D Classification

In production pipelines, a **2D class averaging** step precedes 3D reconstruction:
particles are aligned in-plane and clustered into 2D class averages. This (a) removes
junk/ice particles, (b) validates the dataset is homogeneous enough for 3D work, and
(c) provides the first SNR boost via averaging. This tutorial skips 2D classification
for brevity, but its omission is a simplification of the real workflow.

In [ ]:
# ============================================================
# MODULE 2 — 3D RECONSTRUCTION WITH TRILINEAR GRIDDING
# ============================================================

def euler_to_rotation_matrix(phi, theta, psi):
    """ZYZ Euler angles (degrees) → 3×3 rotation matrix."""
    phi_r, theta_r, psi_r = np.deg2rad(phi), np.deg2rad(theta), np.deg2rad(psi)
    def Rz(a): return np.array([[np.cos(a),-np.sin(a),0],
                                 [np.sin(a), np.cos(a),0],[0,0,1]])
    def Ry(a): return np.array([[ np.cos(a),0,np.sin(a)],
                                 [0,1,0],[-np.sin(a),0,np.cos(a)]])
    return Rz(phi_r) @ Ry(theta_r) @ Rz(psi_r)


def trilinear_insert(F3D, W3D, kx_f, ky_f, kz_f, values, Nbox, em_weight=1.0):
    """
    Vectorized trilinear (8-point) scatter into F3D and W3D.

    FIX (v3): added `em_weight` parameter so the M-step can pass the
    posterior EM weight. Both F3D and W3D are now correctly scaled:
      - F3D accumulates: trilinear_wb × em_weight × F2D_values
      - W3D accumulates: trilinear_wb × em_weight
    This ensures density compensation F3D/W3D gives the correctly
    weighted average rather than a biased ratio.

    Without em_weight (default 1.0) behaviour is identical to before.

    Parameters
    ----------
    kx_f, ky_f, kz_f : (M,) float — fractional Fourier coordinates (NOT scaled by w)
    values            : (M,) complex — Fourier values to insert
    em_weight         : float — EM posterior weight for this (particle, pose) pair
    """
    ix0 = np.floor(kx_f).astype(int) % Nbox
    iy0 = np.floor(ky_f).astype(int) % Nbox
    iz0 = np.floor(kz_f).astype(int) % Nbox
    ix1 = (ix0 + 1) % Nbox
    iy1 = (iy0 + 1) % Nbox
    iz1 = (iz0 + 1) % Nbox

    wx = kx_f - np.floor(kx_f);  ux = 1.0 - wx
    wy = ky_f - np.floor(ky_f);  uy = 1.0 - wy
    wz = kz_f - np.floor(kz_f);  uz = 1.0 - wz

    corners = [
        (ix0,iy0,iz0, ux*uy*uz), (ix1,iy0,iz0, wx*uy*uz),
        (ix0,iy1,iz0, ux*wy*uz), (ix1,iy1,iz0, wx*wy*uz),
        (ix0,iy0,iz1, ux*uy*wz), (ix1,iy0,iz1, wx*uy*wz),
        (ix0,iy1,iz1, ux*wy*wz), (ix1,iy1,iz1, wx*wy*wz),
    ]
    for (bx, by, bz, wb) in corners:
        np.add.at(F3D, (bx, by, bz), wb * values)
        np.add.at(W3D, (bx, by, bz), wb * em_weight)   # ← scale W3D by EM weight


def make_spherical_mask(Nbox, radius_frac=0.45, edge_width=3):
    """
    Soft-edged spherical mask in real space.
    Applied after each reconstruction to suppress solvent noise.
    Without masking, FSC is inflated at low resolution by correlated
    noise — giving falsely optimistic resolution estimates.
    """
    c = np.linspace(-1.0, 1.0, Nbox)
    Z, Y, X = np.meshgrid(c, c, c, indexing='ij')
    r    = np.sqrt(X**2 + Y**2 + Z**2)
    soft = gaussian_filter((r <= radius_frac).astype(float), sigma=edge_width)
    return soft / soft.max()


def reconstruct_fourier_insertion(projections, poses, Nbox, use_trilinear=True):
    """
    3D reconstruction from 2D projections via Direct Fourier Insertion.
    Uses trilinear gridding (8-point scatter) by default.
    Applies a soft spherical mask after IFFT.
    """
    F3D = np.zeros((Nbox,)*3, dtype=complex)
    W3D = np.zeros((Nbox,)*3, dtype=float)

    freq     = np.fft.fftfreq(Nbox) * Nbox
    kx2d, ky2d = np.meshgrid(freq, freq, indexing='ij')
    k2d_flat = np.stack([kx2d.ravel(), ky2d.ravel(), np.zeros(Nbox**2)], axis=1)

    for proj, pose in zip(projections, poses):
        F2D = np.fft.fft2(proj).ravel()
        R   = euler_to_rotation_matrix(*pose)
        k3d = (R @ k2d_flat.T).T

        if use_trilinear:
            trilinear_insert(F3D, W3D,
                             k3d[:,0], k3d[:,1], k3d[:,2],
                             F2D, Nbox)          # em_weight defaults to 1.0
        else:
            ix = np.round(k3d[:,0]).astype(int) % Nbox
            iy = np.round(k3d[:,1]).astype(int) % Nbox
            iz = np.round(k3d[:,2]).astype(int) % Nbox
            np.add.at(F3D, (ix,iy,iz), F2D)
            np.add.at(W3D, (ix,iy,iz), 1.0)

    F3D_comp   = np.where(W3D > 0, F3D / (W3D + 1e-6), 0.0)
    volume_rec = np.real(np.fft.ifftn(F3D_comp))
    return volume_rec * make_spherical_mask(Nbox), F3D, W3D


def filtered_back_projection_2d(projections_1d_set, angles_deg):
    """
    FBP for 2D slice reconstruction.
    Hann window fix: np.hanning(len(ramp)) — symmetric window of correct
    length. Previous code took ascending-from-zero first half, suppressing
    low-frequency signal (opposite of intent).
    """
    n_angles, Nbox = projections_1d_set.shape
    recon = np.zeros((Nbox, Nbox), dtype=float)
    freqs = np.fft.rfftfreq(Nbox)
    ramp  = 2.0 * np.abs(freqs)
    hann  = np.hanning(len(ramp))          # ✅ FIX: symmetric window
    filt  = ramp * hann

    coords_1d = np.linspace(-1, 1, Nbox)
    X, Y = np.meshgrid(coords_1d, coords_1d, indexing='xy')

    for i, angle_deg in enumerate(angles_deg):
        angle_rad = np.deg2rad(angle_deg)
        proj_filt = np.fft.irfft(np.fft.rfft(projections_1d_set[i]) * filt, n=Nbox)
        t         = (X*np.cos(angle_rad) + Y*np.sin(angle_rad))
        t         = ((t + 1) / 2.0) * (Nbox - 1)
        t         = np.clip(t, 0, Nbox - 1)
        t_floor   = np.floor(t).astype(int)
        t_ceil    = np.minimum(t_floor + 1, Nbox - 1)
        w         = t - t_floor
        recon    += proj_filt[t_floor]*(1-w) + proj_filt[t_ceil]*w
    return recon * np.pi / (2.0 * n_angles)


def bfactor_sharpen(volume, pixel_size_A, bfactor=-50.0, lp_resolution_A=None):
    """
    B-factor sharpening for reconstructed volumes.
    weight(s) = exp(B * s²/4)  — negative B sharpens; always pair with
    low-pass filter to avoid boosting noise above resolution limit.
    Uses Nv (not N) to avoid shadowing the global box-size variable.
    """
    Nv  = volume.shape[0]
    FV  = np.fft.fftn(volume)
    f1d = np.fft.fftfreq(Nv, d=pixel_size_A)
    kz, ky, kx  = np.meshgrid(f1d, f1d, f1d, indexing='ij')
    s2           = kx**2 + ky**2 + kz**2
    sharp_filter = np.exp(bfactor * s2 / 4.0)
    if lp_resolution_A is not None:
        lp = gaussian_filter((np.sqrt(s2) <= 1.0/lp_resolution_A).astype(float), sigma=1.5)
        sharp_filter *= lp
    return np.real(np.fft.ifftn(FV * sharp_filter))


def compute_fsc(vol1, vol2):
    """Fourier Shell Correlation between two half-maps."""
    Nv = vol1.shape[0]
    F1, F2 = np.fft.fftn(vol1), np.fft.fftn(vol2)
    f1d = np.fft.fftfreq(Nv)
    kz, ky, kx = np.meshgrid(f1d, f1d, f1d, indexing='ij')
    r_idx = np.round(np.sqrt(kx**2+ky**2+kz**2)*Nv).astype(int)
    fsc   = np.zeros(Nv//2)
    for s in range(Nv//2):
        mask = r_idx == s
        if not mask.any(): continue
        f1s, f2s = F1[mask], F2[mask]
        num = np.real(np.sum(f1s * np.conj(f2s)))
        den = np.sqrt(np.sum(np.abs(f1s)**2) * np.sum(np.abs(f2s)**2))
        fsc[s] = num / den if den > 0 else 0.0
    return fsc, np.arange(Nv//2) / Nv


def compute_ssnr(fsc_curve):
    """
    Spectral Signal-to-Noise Ratio derived from FSC.

    SSNR(s) = 2·FSC(s) / (1 - FSC(s))

    This relation holds under the half-dataset model where each half-map
    has SNR = SSNR/2. It provides a direct estimate of the per-shell SNR
    in the final (full-dataset) map. Resolution is where SSNR ≈ 1.
    Useful alongside FSC for understanding where reconstruction is reliable.
    """
    eps  = 1e-6
    fsc  = np.clip(fsc_curve, -1.0 + eps, 1.0 - eps)
    return 2.0 * fsc / (1.0 - fsc)


def guinier_plot_data(volume, pixel_size_A):
    """
    Compute data for a Guinier plot: log|F(s)| vs s².

    In the Guinier regime the rotationally-averaged Fourier amplitude
    decays as a Gaussian:
        log|F(s)| ≈ log|F(0)| - (B/4)·s²

    Slope of the log-linear region = -B/4, so B = -4 × slope.
    Negative B indicates the map needs sharpening (normal for cryo-EM);
    the fit gives the effective overall B-factor for that map.

    Resolution limit is estimated as the frequency where log|F| deviates
    below the linear fit by more than 2σ of residuals.

    Parameters
    ----------
    volume       : (N,N,N) float — reconstructed map
    pixel_size_A : float — Å per voxel

    Returns
    -------
    s2_bins   : radial s² values (1/Å²)
    logF      : log(mean |F| per shell)
    B_factor  : estimated global B-factor (Å²)
    fit_line  : fitted Guinier line values
    fit_mask  : bool mask for shells used in fit
    """
    Nv  = volume.shape[0]
    FV  = np.fft.fftn(volume)
    f1d = np.fft.fftfreq(Nv, d=pixel_size_A)
    kz, ky, kx = np.meshgrid(f1d, f1d, f1d, indexing='ij')
    r_idx = np.round(np.sqrt(kx**2+ky**2+kz**2)*Nv).astype(int)

    shells   = np.arange(1, Nv//2)
    s_vals   = shells / (Nv * pixel_size_A)   # 1/Å
    s2_bins  = s_vals**2
    logF     = np.zeros(len(shells))

    for i, s in enumerate(shells):
        mask = r_idx == s
        if mask.any():
            logF[i] = np.log(np.mean(np.abs(FV[mask])) + 1e-30)

    # Fit in Guinier region: s < 1/(2*pixel_size_A * 4) ~ low-freq regime
    fit_mask = (s2_bins > 0) & (s2_bins < (1/15.0)**2)   # up to ~15 Å resolution
    B_factor, fit_line = 0.0, logF
    if fit_mask.sum() > 3:
        slope, intercept = np.polyfit(s2_bins[fit_mask], logF[fit_mask], 1)
        B_factor  = -4.0 * slope
        fit_line  = slope * s2_bins + intercept

    return s2_bins, logF, B_factor, fit_line, fit_mask


# ── Run all reconstructions ────────────────────────────────────
print("🔄 Running reconstructions...")

vol_trilinear, F3D, W3D = reconstruct_fourier_insertion(
    projections_wiener, true_poses, N, use_trilinear=True)

vol_nn, _, _ = reconstruct_fourier_insertion(
    projections_wiener, true_poses, N, use_trilinear=False)

vol_sharpened = bfactor_sharpen(vol_trilinear, pixel_size_A=3.0,
                                 bfactor=-80.0, lp_resolution_A=9.0)

# FBP sinogram
n_fbp_angles = 36
fbp_angles   = np.linspace(0, 180, n_fbp_angles, endpoint=False)
sinogram     = np.zeros((n_fbp_angles, N))
for i, ang in enumerate(fbp_angles):
    sinogram[i] = rotate(phantom[N//2,:,:], ang, reshape=False).sum(axis=0)
recon_fbp = filtered_back_projection_2d(sinogram, fbp_angles)

# Half-maps FSC
vol_h1,_,_ = reconstruct_fourier_insertion(projections_wiener[0::2], true_poses[0::2], N)
vol_h2,_,_ = reconstruct_fourier_insertion(projections_wiener[1::2], true_poses[1::2], N)
fsc_half,  freq_shells = compute_fsc(vol_h1, vol_h2)
fsc_cross, _           = compute_fsc(phantom, vol_trilinear)

# SSNR from half-map FSC
ssnr_curve = compute_ssnr(fsc_half)

# Guinier plot data
s2_bins, logF_tri, B_est, fit_line, fit_mask = guinier_plot_data(vol_trilinear, pixel_size_A=3.0)
_, logF_gt, _, _, _ = guinier_plot_data(phantom, pixel_size_A=3.0)

r_tri = pearsonr(phantom.ravel(), vol_trilinear.ravel())[0]
r_nn  = pearsonr(phantom.ravel(), vol_nn.ravel())[0]
print(f"✅ Done.")
print(f"   Trilinear recon Pearson r = {r_tri:.4f}")
print(f"   NN recon        Pearson r = {r_nn:.4f}  (should be lower)")
print(f"   Estimated B-factor (trilinear map): {B_est:.1f} Å²")


In [ ]:
# ============================================================
# VISUALIZATION 2A — Trilinear vs NN Gridding Comparison
# FIX (v3): Ground truth column now shows '—' instead of 'r=0.000'.
#           The previous code returned (0,0) for col==0 which made
#           the ground-truth panel falsely appear uncorrelated.
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Module 2 — Reconstruction Quality: Trilinear vs Nearest-Neighbour',
             fontsize=13, fontweight='bold', color='#58a6ff')

vols   = [phantom,    vol_nn,       vol_trilinear,    vol_sharpened]
titles = ['Ground Truth', 'NN gridding', 'Trilinear gridding', 'Trilinear + B-sharp']
colors = ['#8b949e', '#ff7b72',  '#3fb950',           '#79c0ff']

for ci, (vol, tt, col) in enumerate(zip(vols, titles, colors)):
    for ri, (plane, ref_sl, sl) in enumerate([
        ('XY', phantom[N//2,:,:], vol[N//2,:,:]),
        ('XZ', phantom[:,N//2,:], vol[:,N//2,:]),
    ]):
        ax = axes[ri, ci]
        ax.imshow(sl, cmap='inferno')
        if ci == 0:
            # Ground truth: show plane label only, no self-correlation
            ax.set_title(f'Ground Truth\n({plane})', color=col, fontsize=8, fontweight='bold')
        else:
            r, _ = pearsonr(ref_sl.ravel(), sl.ravel())
            ax.set_title(f'{tt}\n{plane} | r={r:.3f}', color=col, fontsize=8, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(col); sp.set_linewidth(2.0)

plt.tight_layout(); plt.show()

# Residual maps
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
fig2.suptitle('Residual Maps (Ground Truth − Reconstruction)',
              fontsize=12, fontweight='bold', color='#58a6ff')
for ci, (vol, tt, col) in enumerate(zip(
        [vol_nn, vol_trilinear, vol_sharpened],
        ['NN', 'Trilinear', 'Trilinear+B-sharp'],
        ['#ff7b72', '#3fb950', '#79c0ff'])):
    diff = phantom[N//2,:,:] - vol[N//2,:,:]
    vm   = np.abs(diff).max()
    im   = axes2[ci].imshow(diff, cmap='RdBu_r', vmin=-vm, vmax=vm)
    r, _ = pearsonr(phantom[N//2,:,:].ravel(), vol[N//2,:,:].ravel())
    axes2[ci].set_title(f'{tt} | r={r:.4f}', color=col, fontsize=9)
    axes2[ci].set_xticks([]); axes2[ci].set_yticks([])
    plt.colorbar(im, ax=axes2[ci], fraction=0.046)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# VISUALIZATION 2B — Sinogram, FBP, FSC, SSNR & Guinier Plot
# NEW (v3): added SSNR panel and Guinier plot panel
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Module 2 — FBP, Fourier Shell Correlation, SSNR & Guinier Plot',
             fontsize=13, fontweight='bold', color='#58a6ff')

# ── Row 0: Sinogram, FBP, GT, Fourier weight map ──────────────
im = axes[0,0].imshow(sinogram, aspect='auto', cmap='gray')
axes[0,0].set_title('Sinogram (36 angles)', color='#e6edf3')
axes[0,0].set_xlabel('Detector pos'); axes[0,0].set_ylabel('Angle index')
plt.colorbar(im, ax=axes[0,0], fraction=0.046)

im = axes[0,1].imshow(recon_fbp, cmap='inferno')
axes[0,1].set_title('FBP reconstruction (central XZ)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,1], fraction=0.046)

im = axes[0,2].imshow(phantom[N//2,:,:], cmap='inferno')
axes[0,2].set_title('Ground truth (XZ)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,2], fraction=0.046)

im = axes[0,3].imshow(np.log1p(fftshift(W3D)[N//2,:,:]), cmap='plasma')
axes[0,3].set_title('3D Fourier sampling\n(log weight, central slice)', color='#e6edf3')
plt.colorbar(im, ax=axes[0,3], fraction=0.046, label='log(1+count)')

# ── Row 1: FSC, SSNR, Guinier, FSC map ────────────────────────
freq_A = freq_shells / 3.0   # convert to 1/Å

# Panel: FSC curves
ax = axes[1,0]
ax.plot(freq_A, fsc_half,  color='#79c0ff', lw=2, label='Half-map FSC')
ax.plot(freq_A, fsc_cross, color='#f0883e', lw=2, label='Cross FSC (vs GT)')
ax.axhline(0.143, color='#ff7b72', ls='--', lw=1.5, label='FSC=0.143 (gold std)')
ax.axhline(0.500, color='#8957e5', ls='--', lw=1.5, label='FSC=0.5')
ax.set_title('Fourier Shell Correlation', color='#e6edf3')
ax.set_xlabel('Spatial freq (1/Å)'); ax.set_ylabel('FSC')
ax.legend(fontsize=7); ax.grid(True)
ax.set_ylim([-0.1, 1.05]); ax.set_xlim([0, freq_A[N//2-1]])

# Panel: SSNR
# NEW (v3): SSNR = 2·FSC / (1 - FSC); resolution where SSNR ~ 1
ax = axes[1,1]
ssnr_plot = np.clip(ssnr_curve, 0, 20)    # cap for display clarity
ax.semilogy(freq_A, ssnr_plot + 1e-3, color='#3fb950', lw=2, label='SSNR (from half-map FSC)')
ax.axhline(1.0,  color='#ff7b72', ls='--', lw=1.5, label='SSNR=1 (resolution limit)')
ax.axhline(0.5,  color='#8957e5', ls='--', lw=1.2, label='SSNR=0.5')
ax.set_title('Spectral SNR\nSSNR = 2·FSC/(1−FSC)', color='#e6edf3')
ax.set_xlabel('Spatial freq (1/Å)'); ax.set_ylabel('SSNR (log scale)')
ax.legend(fontsize=7); ax.grid(True)
ax.set_xlim([0, freq_A[N//2-1]])
ax.fill_between(freq_A, 0, ssnr_plot + 1e-3,
                where=ssnr_plot >= 1.0, alpha=0.15, color='#3fb950',
                label='Well-resolved (SSNR>1)')

# Panel: Guinier plot
# NEW (v3): log|F| vs s² — slope gives B-factor
ax = axes[1,2]
ax.plot(s2_bins, logF_gt,  color='#8b949e', lw=1.5, alpha=0.7, label='Ground truth')
ax.plot(s2_bins, logF_tri, color='#3fb950',  lw=2,  label='Trilinear recon')
ax.plot(s2_bins[fit_mask], fit_line[fit_mask],
        color='#ff7b72', lw=2.5, ls='--', label=f'Guinier fit (B≈{B_est:.0f} Å²)')
ax.set_title(f'Guinier Plot\nlog|F| vs s² — B-factor = {B_est:.1f} Å²',
             color='#e6edf3')
ax.set_xlabel('s² (1/Å²)'); ax.set_ylabel('log |F(s)|')
ax.legend(fontsize=7); ax.grid(True)
ax.set_xlim([0, s2_bins[N//2-2]])
ax.annotate(f'Slope = −B/4 = {-B_est/4:.1f}\n→ B = {B_est:.0f} Å²',
            xy=(s2_bins[fit_mask].mean(), fit_line[fit_mask].mean()),
            xytext=(s2_bins[fit_mask].max()*1.1, fit_line[fit_mask].min()),
            color='#ff7b72', fontsize=8,
            arrowprops=dict(arrowstyle='->', color='#ff7b72', lw=1.2))

# Panel: FSC map
ax = axes[1,3]
fsc_vol = np.zeros((N,)*3)
f1d = np.fft.fftfreq(N)
kz,ky,kx = np.meshgrid(f1d,f1d,f1d,indexing='ij')
r_idx = np.clip(np.round(np.sqrt(kx**2+ky**2+kz**2)*N).astype(int), 0, N//2-1)
for s in range(N//2):
    fsc_vol[r_idx==s] = fsc_cross[s]
im = ax.imshow(fftshift(fsc_vol)[N//2,:,:], cmap='RdYlGn', vmin=-0.1, vmax=1.0)
ax.set_title('Cross-FSC map\n(central slice; green=resolved)', color='#e6edf3')
plt.colorbar(im, ax=ax, fraction=0.046, label='FSC')

plt.tight_layout(); plt.show()


---
# Module 3 — Pose Estimation via Expectation-Maximization

## 3.1 The Latent Pose Problem

In real cryo-EM, particle orientations are **completely unknown**. The EM algorithm
treats poses as latent variables and iterates between:

- **E-step:** For each particle, compute soft posterior probability over all
  candidate poses $R_j$ from a reference library
- **M-step:** Update the 3D volume using the weighted sum over all particles and poses

## 3.2 E-step: Normalized Cross-Correlation

Rather than a direct L2 distance (slow Python loops), the **normalized cross-correlation (NCC)**
via FFT provides:

1. **Computational efficiency**: vectorized via $\mathcal{F}_2$, $O(N^2 \log N)$ vs $O(N^2)$
   with Python overhead
2. **Translation marginalization**: `cc.max()` implicitly picks the best in-plane shift —
   the NCC peak position gives the optimal 2D translation, and its value gives the
   similarity score for this pose. This means each E-step score already marginalizes
   over all in-plane translations, which is exactly the correct probabilistic treatment.

## 3.3 M-step and FSC Monitoring

The reconstructed half-maps at each iteration can be FSC-compared to track convergence.
The ELBO (evidence lower bound) should increase monotonically if the model is correct.

## 3.4 Production Methods

| Method | Approach | Notes |
|--------|----------|-------|
| RELION | EM + healpix SO(3) sampling | Gold standard |
| cryoSPARC | Branch-and-bound + SGD | Fast, scalable |
| cryoDRGN | Amortised VAE encoder | Heterogeneous |

In [ ]:
# ============================================================
# MODULE 3 — EM POSE REFINEMENT (NCC + fixed log-sum-exp)
# ============================================================

def generate_reference_projections(volume, pose_grid):
    """Pre-compute projection library for E-step template matching."""
    return np.array([project_volume(volume, pose) for pose in pose_grid])


def ncc_score(image, template):
    """
    Normalised cross-correlation score via FFT.

    Returns cc.max() — the peak NCC value after marginalising over
    all in-plane 2D translations. The peak position gives the optimal
    shift; the peak value is the score for this (pose, shift) pair.
    This is the correct probabilistic treatment for an EM E-step.

    Complexity: O(N² log N), vectorized — much faster than direct
    L2 distance with Python loops.
    """
    F_img = np.fft.fft2(image    - image.mean())
    F_ref = np.fft.fft2(template - template.mean())
    cc    = np.real(np.fft.ifft2(F_img * np.conj(F_ref)))
    norm  = image.std() * template.std() * image.size
    return float(cc.max() / (norm + 1e-8))


def e_step(observed_images, reference_projections, sigma=1.0, use_ncc=True):
    """
    E-Step: compute soft pose assignment weights per particle.

    Log-sum-exp fix (v2): now correctly: log_max + log(Σ exp(x - log_max))
    NCC mode: ncc_score / sigma as log-likelihood proxy, marginalized over
    in-plane translations. Enables GPU-batching in production code.
    """
    n_particles = len(observed_images)
    n_poses     = len(reference_projections)
    scores      = np.zeros((n_particles, n_poses))

    for i, img in enumerate(observed_images):
        for j, ref in enumerate(reference_projections):
            if use_ncc:
                scores[i,j] = ncc_score(img, ref) / sigma
            else:
                diff        = img - ref
                scores[i,j] = -np.sum(diff**2) / (2 * sigma**2)

    # Numerically stable softmax
    scores_stable = scores - scores.max(axis=1, keepdims=True)
    weights       = np.exp(scores_stable)
    weights      /= weights.sum(axis=1, keepdims=True)

    # ✅ Correct log-sum-exp (ELBO estimate)
    log_max      = scores.max(axis=1)
    log_evidence = np.mean(
        log_max + np.log(np.sum(np.exp(scores - log_max[:,None]), axis=1))
    )
    return weights, log_evidence


def m_step(observed_images, reference_poses, weights, Nbox):
    """
    M-Step: weighted Fourier insertion using posterior pose probabilities.
    Uses trilinear gridding for accurate interpolation.

    FIX (v3): Two bugs corrected from v2:
    ──────────────────────────────────────────────────────────
    BUG 1 — Coordinates scaled by w:
      v2 passed k3d[:,0]*w, k3d[:,1]*w, k3d[:,2]*w to trilinear_insert.
      When w ≈ 0.01–0.3, all Fourier coordinates are collapsed toward
      the origin — completely wrong grid positions, corrupting F3D.
      FIX: pass k3d[:,0], k3d[:,1], k3d[:,2] unmodified.

    BUG 2 — W3D not weighted by EM posterior:
      v2 W3D accumulated only the trilinear corner weights (wb), not
      scaled by the EM weight w. Since F3D accumulates wb×w×F2D,
      the density compensation F3D/(W3D+ε) gave wb×w×F2D / wb = w×F2D
      instead of the correct weighted average. This biased all shells
      towards high-w particles regardless of Fourier coverage.
      FIX: pass em_weight=w to trilinear_insert so W3D accumulates wb×w.
    ──────────────────────────────────────────────────────────
    """
    F3D = np.zeros((Nbox,)*3, dtype=complex)
    W3D = np.zeros((Nbox,)*3, dtype=float)
    freq = np.fft.fftfreq(Nbox) * Nbox
    kx2d, ky2d = np.meshgrid(freq, freq, indexing='ij')
    k2d_flat = np.stack([kx2d.ravel(), ky2d.ravel(), np.zeros(Nbox**2)], axis=1)

    for i, img in enumerate(observed_images):
        F2D = np.fft.fft2(img).ravel()
        for j, pose in enumerate(reference_poses):
            w = weights[i, j]
            if w < 1e-6:
                continue
            R   = euler_to_rotation_matrix(*pose)
            k3d = (R @ k2d_flat.T).T
            # ✅ FIX: unscaled coordinates + em_weight for correct W3D
            trilinear_insert(F3D, W3D,
                             k3d[:,0], k3d[:,1], k3d[:,2],   # NOT k3d*w
                             w * F2D, Nbox, em_weight=w)     # W3D scaled by w

    F3D_comp = np.where(W3D > 0, F3D / (W3D + 1e-6), 0.0)
    vol = np.real(np.fft.ifftn(F3D_comp))
    return vol * make_spherical_mask(Nbox)


# ── EM loop ────────────────────────────────────────────────────
coarse_poses = [
    (  0,  0,  0),( 45, 0, 0),( 90,  0, 0),(135,  0,  0),
    (  0, 45,  0),( 45,45, 0),( 90, 45, 0),(135, 45,  0),
    (  0, 90,  0),( 45,90, 0),( 90, 90, 0),(135, 90,  0),
    (  0,135,  0),( 45,135,0),( 90,135, 0),(135,135,  0),
]

current_vol     = gaussian_filter(phantom, sigma=4.0)
demo_particles  = projections_wiener[:3]   # Wiener-corrected inputs
n_em_iters      = 3
em_volumes      = [current_vol.copy()]
em_weights_hist = []
em_log_evidence = []

print("🔄 EM pose refinement (NCC-based E-step, trilinear M-step)...\n")
for it in range(n_em_iters):
    ref_projs = generate_reference_projections(current_vol, coarse_poses)
    weights, log_ev = e_step(demo_particles, ref_projs, sigma=0.3, use_ncc=True)
    em_log_evidence.append(log_ev)
    best = np.argmax(weights, axis=1)
    current_vol = m_step(demo_particles, coarse_poses, weights, N)
    em_volumes.append(current_vol.copy())
    em_weights_hist.append(weights.copy())
    print(f"  Iter {it+1}/{n_em_iters} | ELBO: {log_ev:.3f} | "
          f"Best poses: " + " / ".join([f"#{best[p]}" for p in range(3)]))

print("\n✅ EM complete.")


In [ ]:
# ============================================================
# VISUALIZATION 3 — EM weights, volume evolution, FSC
# ============================================================
fig = plt.figure(figsize=(20, 10))
fig.suptitle('Module 3 — EM Pose Refinement',
             fontsize=14, fontweight='bold', color='#58a6ff')
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.35)

# Weight heatmaps
for it_idx,w in enumerate(em_weights_hist):
    ax = fig.add_subplot(gs[0, it_idx])
    im = ax.imshow(w, aspect='auto', cmap='YlOrRd', vmin=0, vmax=w.max())
    ax.set_title(f'Iter {it_idx+1}\nPosterior weights', color='#e6edf3', fontsize=9)
    ax.set_xlabel('Pose index', fontsize=8)
    ax.set_ylabel('Particle', fontsize=8)
    ax.set_yticks(range(len(demo_particles)))
    ax.set_yticklabels([f'P{i}' for i in range(len(demo_particles))], fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)

# ELBO convergence
ax_ev = fig.add_subplot(gs[0, 3])
ax_ev.plot(range(1, n_em_iters+1), em_log_evidence,
           'o-', color='#3fb950', lw=2, ms=8)
ax_ev.fill_between(range(1, n_em_iters+1), em_log_evidence, alpha=0.2, color='#3fb950')
ax_ev.set_title('ELBO convergence\n(correct log-sum-exp)', color='#e6edf3', fontsize=9)
ax_ev.set_xlabel('Iteration'); ax_ev.set_ylabel('ELBO')
ax_ev.grid(True)

# Volume evolution
for col, (vol, lab) in enumerate(zip(em_volumes,
        ['Init (blurred)','After iter 1','After iter 2','After iter 3'])):
    ax = fig.add_subplot(gs[1, col])
    sl = vol[N//2,:,:]
    im = ax.imshow(sl, cmap='inferno')
    r,_ = pearsonr(phantom[N//2,:,:].ravel(), sl.ravel())
    ax.set_title(f'{lab}\nr={r:.3f}', color='#f0883e', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.show()

---
# Module 4 — Neural Rendering for Conformational Heterogeneity

## 4.1 The Heterogeneity Problem

Classical cryo-EM assumes a **homogeneous, rigid** particle. Real biology is far messier:
continuous conformational landscapes (flexible loops, breathing motions), compositional
variation (missing subunits), and discrete multi-state equilibria.

Reconstructing from a heterogeneous dataset with a single-model pipeline produces a
**blurred average** — structural detail is lost precisely where it matters most.

## 4.2 cryoDRGN: VAE + Implicit Neural Representation

**cryoDRGN** (Zhong et al., *Nature Methods* 2021) encodes each particle image into a
continuous latent vector $z \in \mathbb{R}^d$, then decodes $(x, z) \mapsto \rho_z(x)$
via a coordinate MLP (implicit neural representation / INR):

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q(z|I_i)}[\log p(I_i|z,R_i)] - \beta \cdot D_{\text{KL}}[q(z|I_i)\|p(z)]$$

The reconstruction term is evaluated in **Fourier space** (after CTF weighting), which
is numerically stable and naturally handles the projection geometry.

## 4.3 3DFlex: Physically Constrained Deformation Fields

**3DFlex** (Punjani & Fleet, *Nature Methods* 2023) predicts per-voxel 3D displacement
fields from $z$:

$$\rho_z(\mathbf{x}) = \rho_0(\mathbf{x} + \Delta\mathbf{x}_z(\mathbf{x}))$$

Physical constraints (volume conservation, local rigidity) make the latent dimensions
directly interpretable as physical motions — not abstract activations.

## 4.4 Architecture Comparison

| Method | Latent Space | Physical Priors | Strengths |
|--------|-------------|-----------------|-----------|
| cryoDRGN | Continuous $\mathbb{R}^d$ | None | General, flexible |
| 3DFlex | Continuous $\mathbb{R}^d$ | Volume, rigidity | Interpretable |
| RECOVAR | PCA manifold | Linear | Fast, reproducible |
| DynaMight | Gaussian mixture | Soft | Multi-resolution |
| CryoFIRE | Implicit NR | None | Pose-free training |

## 4.5 Latent Space Traversal

Linear interpolation $z(t) = (1-t)z_0 + t z_1$ generates morphing movies.
PCA or UMAP on $\{z_i\}$ across all particles reveals the **conformational
manifold**, often exposing intermediate states invisible to discrete 3D classification.

In [ ]:
# ============================================================
# MODULE 4 — CONFORMATIONAL HETEROGENEITY & NEURAL RENDERING
# ============================================================

def build_conformational_phantom(Nb, z1, z2):
    """
    Parametric phantom where latent codes (z1, z2) directly control
    physical degrees of freedom:
      z1 ∈ [-1,1] → arm rotation angle  (interprets as a hinge motion)
      z2 ∈ [-1,1] → subunit opening gap (interprets as binding cleft)
    """
    c = np.linspace(-1.0, 1.0, Nb)
    Z,Y,X = np.meshgrid(c, c, c, indexing='ij')
    def g(x0,y0,z0,sx,sy,sz,a=1.0):
        return a*np.exp(-0.5*(((X-x0)/sx)**2+((Y-y0)/sy)**2+((Z-z0)/sz)**2))

    vol  = g( 0.0,  0.0,  0.0, 0.30, 0.30, 0.28, 1.0)   # rigid core
    vol += g(0.55*np.cos(z1*np.pi*0.6),
              0.55*np.sin(z1*np.pi*0.6), 0.1, 0.12,0.12,0.18, 0.6)  # arm
    vol += g(-0.30, +z2*0.5, 0.0, 0.15,0.15,0.15, 0.5)  # lobe A
    vol += g(-0.30, -z2*0.5, 0.0, 0.15,0.15,0.15, 0.5)  # lobe B
    vol = (vol-vol.min())/(vol.max()-vol.min())
    return vol


def simple_inr_decode(z_vec, coords_2d):
    """
    Analytical stand-in for a SIREN/MLP INR decoder.
    In production: MLP with sinusoidal activations, Fourier feature
    embeddings, or hash-grid encodings.
    f_θ(x, z) → ρ_z(x)  — density at coordinate x conditioned on z.
    """
    x, y = coords_2d[...,0], coords_2d[...,1]
    z1, z2 = z_vec[0], z_vec[1]
    return (
        np.exp(-3*(x**2+y**2)) +
        0.5*np.exp(-8*((x-0.4*np.cos(z1*np.pi))**2+(y-0.4*np.sin(z1*np.pi))**2)) +
        0.4*np.exp(-8*((x+0.35)**2+(y-z2*0.3)**2)) +
        0.4*np.exp(-8*((x+0.35)**2+(y+z2*0.3)**2))
    )


N_conf = 64
np.random.seed(7)
latent_true = np.random.randn(8,2) * 0.7
conf_volumes = [build_conformational_phantom(N_conf, lz[0], lz[1])
                for lz in latent_true]

g2 = np.linspace(-1,1,N_conf)
X2d,Y2d = np.meshgrid(g2, g2, indexing='xy')
coords_2d = np.stack([X2d,Y2d], axis=-1)

n_frames   = 9
z1_path    = np.linspace(-1.0, 1.0, n_frames)
latent_path = np.stack([z1_path, np.zeros(n_frames)], axis=1)
decoded_frames = [simple_inr_decode(z, coords_2d) for z in latent_path]

# Simulated particle latent ensemble (3 dominant conformations)
n_sim = 400
centres = np.array([[-0.8,-0.5],[0.5,0.6],[0.2,-0.9]])
sigmas  = [0.25,0.30,0.20]; probs = [0.40,0.35,0.25]
latent_sim, cluster_ids = [], []
for _ in range(n_sim):
    k = np.random.choice(3, p=probs)
    latent_sim.append(centres[k] + np.random.randn(2)*sigmas[k])
    cluster_ids.append(k)
latent_sim  = np.array(latent_sim)
cluster_ids = np.array(cluster_ids)
print(f"✅ Conformational ensemble: {len(conf_volumes)} particles, {n_frames}-frame traversal")

In [ ]:
# ============================================================
# VISUALIZATION 4A — Heterogeneous particle ensemble
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Module 4 — Conformational Heterogeneity (8 Particles)\n'
             'z₁ controls arm angle, z₂ controls subunit gap',
             fontsize=13, fontweight='bold', color='#58a6ff')
for idx,(vol,lz) in enumerate(zip(conf_volumes, latent_true)):
    ax = axes[idx//4, idx%4]
    ax.imshow(vol[N_conf//2,:,:], cmap='inferno', vmin=0, vmax=1)
    ax.set_title(f'P{idx+1}: z₁={lz[0]:+.2f}, z₂={lz[1]:+.2f}',
                 color='#e6edf3', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# ============================================================
# VISUALIZATION 4B — INR latent space traversal
# ============================================================
fig, axes = plt.subplots(1, n_frames, figsize=(22,3))
fig.suptitle('INR Latent Traversal: z₁ ∈ [−1, +1], z₂ = 0  (simulates arm rotation)',
             fontsize=12, fontweight='bold', color='#58a6ff')
for i,(ax,frame,z) in enumerate(zip(axes, decoded_frames, latent_path)):
    ax.imshow(frame, cmap='magma', vmin=0, interpolation='bilinear')
    ax.set_title(f'z₁={z[0]:+.2f}', fontsize=8, color='#8b949e')
    ax.set_xticks([]); ax.set_yticks([])
    col = plt.cm.cool(i/(n_frames-1))
    for sp in ax.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(2.5)
plt.tight_layout(); plt.show()

# ============================================================
# VISUALIZATION 4C — Latent space landscape
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14,6))
fig.suptitle('cryoDRGN-Style Latent Space', fontsize=13,
             fontweight='bold', color='#58a6ff')

colors_k = ['#79c0ff','#3fb950','#f0883e']
labels_k = ['State A (open)','State B (intermediate)','State C (closed)']
ax = axes[0]
for k,(col,lab) in enumerate(zip(colors_k, labels_k)):
    m = cluster_ids==k
    ax.scatter(latent_sim[m,0], latent_sim[m,1],
               color=col, alpha=0.45, s=18, edgecolors='none', label=lab)
    ax.scatter(*centres[k], color=col, s=220, zorder=5,
               marker='*', edgecolors='white', linewidths=0.5)
trav = np.stack([np.linspace(centres[0][0],centres[1][0],50),
                  np.linspace(centres[0][1],centres[1][1],50)],axis=1)
ax.plot(trav[:,0], trav[:,1], 'w--', lw=1.5, alpha=0.7, label='Traversal path')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_title('Particle latent codes', color='#e6edf3')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

h,xe,ye = np.histogram2d(latent_sim[:,0], latent_sim[:,1], bins=30)
im = axes[1].imshow(gaussian_filter(h.T,1.0), aspect='auto', origin='lower',
                     extent=[xe[0],xe[-1],ye[0],ye[-1]], cmap='hot')
for k,(col,lab) in enumerate(zip(colors_k, labels_k)):
    axes[1].annotate(lab.split()[1], xy=centres[k], color=col,
                     fontsize=9, fontweight='bold', ha='center',
                     xytext=(0,12), textcoords='offset points')
axes[1].set_title('Conformational landscape\n(particle density)', color='#e6edf3')
axes[1].set_xlabel('z₁'); axes[1].set_ylabel('z₂')
plt.colorbar(im, ax=axes[1], fraction=0.046, label='Count')
plt.tight_layout(); plt.show()

---
# Summary

## Bugs Fixed (v2 + v3 combined)

| Issue | Location | Fix | Version |
|-------|----------|-----|---------|
| Hann window wrong half | `filtered_back_projection_2d` | `np.hanning(len(ramp))` | v2 |
| Log-sum-exp incorrect | EM `e_step` | `log_max + log(Σ exp(x−log_max))` | v2 |
| `pearsonr` inline import | Module 2 viz cell | Moved to global imports | v2 |
| CTF `indexing='xy'` | `compute_ctf` | Changed to `'ij'` for Fourier alignment | v2 |
| `bfactor_sharpen` shadows `N` | Module 2 | Renamed local variable to `Nv` | v2 |
| **`m_step` multiplies coordinates by EM weight** | `m_step` | Pass `k3d` unscaled; use `em_weight` param | **v3** |
| **`W3D` not scaled by EM weight** | `trilinear_insert` / `m_step` | Added `em_weight` param; W3D accumulates `wb×w` | **v3** |
| **Viz 1A broken loop + `if False` dead code** | Visualization 1A | Removed; rewrote as clean 4-row GridSpec | **v3** |
| **Ground truth shown as `r=0.000`** | Visualization 2A | GT column now shows label only, no misleading r | **v3** |

## Improvements & Additions

| Item | Where | Notes |
|------|-------|-------|
| Trilinear gridding (vectorized 8-point) | Module 2 | Better reconstruction fidelity vs NN |
| NCC-based E-step (FFT, translation-marginalized) | Module 3 | Faster & more correct than L2 loops |
| 3D soft spherical mask | Modules 2 & 3 | Prevents FSC inflation from solvent noise |
| CTF Wiener + phase-flip correction (Module 1.5) | Module 1.5 | Critical step; absent in original v1 |
| Anisotropic CTF with astigmatism | Module 1 | Realistic forward model |
| Dose weighting (`s^{-1.665}`, not `s^{-0.5}`) | Module 1 | Correct Grant-Grigorieff 2015 |
| B-factor sharpening | Module 2 | Standard post-processing |
| **2D class averaging (K-means + NCC alignment)** | **Module 1.8** | Previously described but not implemented |
| **SSNR: `2·FSC/(1−FSC)`** | **Module 2 Viz 2B** | Derived per-shell SNR alongside FSC |
| **Guinier plot** | **Module 2 Viz 2B** | log|F| vs s² to estimate B-factor from map |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Projection | $\mathcal{O}(N^3)$ | Memory bandwidth |
| CTF application | $\mathcal{O}(N^2 \log N)$ | Negligible |
| Trilinear insertion | $\mathcal{O}(P \cdot 8N^2)$ | Parallelisable |
| E-step (NCC) | $\mathcal{O}(P \cdot K \cdot N^2 \log N)$ | Dominant cost |
| cryoDRGN decode | $\mathcal{O}(N^3 \cdot L \cdot H)$ | GPU compute |

## Key References

- Crowther (1970) — projection theorem & angular sampling
- Frank (2006) — *Three-Dimensional Electron Microscopy of Macromolecular Assemblies*
- Grant & Grigorieff (2015) — dose weighting (*eLife*)
- Scheres (2012) — RELION Bayesian framework (*J. Struct. Biol.*)
- Punjani et al. (2017) — cryoSPARC (*Nature Methods*)
- Zhong et al. (2021) — cryoDRGN (*Nature Methods*)
- Punjani & Fleet (2023) — 3DFlex (*Nature Methods*)


In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# FIX (v3): GT panel shows title without false r=0.000
# ============================================================
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🔬 Cryo-EM 3D Reconstruction — Complete Pipeline Dashboard (v3)',
             fontsize=16, fontweight='bold', color='#58a6ff', y=0.98)
gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.55, wspace=0.32)

# ── Row 0: Forward model ──────────────────────────────────────
row0 = [
    (phantom[N//2,:,:],         'M1: Phantom XY',        'inferno'),
    (projections_clean[2],      'M1: Clean proj',          'gray'),
    (projections_noisy[2],      'M1: + CTF + Noise',       'gray'),
    (projections_wiener[2],     'M1.5: Wiener corrected',  'gray'),
    (fftshift(CTF_2D**2),       'M1: CTF² (Thon rings)', 'viridis'),
]
for col, (data, title, cm) in enumerate(row0):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(data, cmap=cm)
    ax.set_title(title, color='#79c0ff', fontsize=8, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

# ── Row 1: Reconstruction ─────────────────────────────────────
row1_data = [
    (np.log1p(fftshift(W3D)[N//2,:,:]), 'M2: Fourier sampling', 'plasma',   None),
    (vol_nn[N//2,:,:],                   'M2: NN gridding',      'inferno',  vol_nn),
    (vol_trilinear[N//2,:,:],            'M2: Trilinear',        'inferno',  vol_trilinear),
    (vol_sharpened[N//2,:,:],            'M2: +B-sharpening',    'inferno',  vol_sharpened),
]
for col, (data, title, cm, vol) in enumerate(row1_data):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(data, cmap=cm)
    if vol is not None:
        r, _ = pearsonr(phantom[N//2,:,:].ravel(), data.ravel())
        ax.set_title(f'{title}\nr={r:.3f}', color='#3fb950', fontsize=8, fontweight='bold')
    else:
        ax.set_title(title, color='#3fb950', fontsize=8, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

ax_fsc = fig.add_subplot(gs[1, 4])
ax_fsc.plot(freq_shells/3.0, fsc_cross, color='#f0883e', lw=2, label='Cross-FSC')
ax_fsc.semilogy(freq_shells/3.0, np.clip(ssnr_curve,1e-2,20),
                color='#3fb950', lw=1.5, ls=':', label='SSNR')
ax_fsc.axhline(0.143, color='#ff7b72', ls='--', lw=1.5, label='FSC=0.143')
ax_fsc.set_title('M2: FSC + SSNR', color='#3fb950', fontsize=8, fontweight='bold')
ax_fsc.set_xlabel('1/Å', fontsize=7); ax_fsc.grid(True)
ax_fsc.legend(fontsize=6); ax_fsc.set_ylim([-0.1, 20])

# ── Row 2: EM + latent space ──────────────────────────────────
for col, (vol, lab) in enumerate(zip(em_volumes[:4],
        ['M3: Init', 'M3: Iter1', 'M3: Iter2', 'M3: Iter3'])):
    ax = fig.add_subplot(gs[2, col])
    sl = vol[N//2,:,:]
    ax.imshow(sl, cmap='inferno')
    r, _ = pearsonr(phantom[N//2,:,:].ravel(), sl.ravel())
    ax.set_title(f'{lab}\nr={r:.3f}', color='#f0883e', fontsize=8, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

ax_lat = fig.add_subplot(gs[2, 4])
for k, col in enumerate(['#79c0ff', '#3fb950', '#f0883e']):
    m = cluster_ids == k
    ax_lat.scatter(latent_sim[m,0], latent_sim[m,1],
                   color=col, alpha=0.4, s=10, edgecolors='none')
ax_lat.set_title('M4: Latent space', color='#ff7b72', fontsize=8, fontweight='bold')
ax_lat.set_xlabel('z₁', fontsize=7); ax_lat.set_ylabel('z₂', fontsize=7)
ax_lat.grid(True, alpha=0.3)

from pathlib import Path
out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'cryoem_dashboard_v3.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ Dashboard saved to {out_path.resolve()}")
